# Deliverable 4 — Demo en Vivo: Control Base vs Sistema Oficial `3-LoRA` (<= 3 min)
**Proyecto:** Especialización Multi-LoRA por rol (`Enrutador + Recuperador + Redactor` QLoRA con `set_adapter()`, `Verificador` Base con `disable_adapter()`) sobre `Qwen/Qwen3-4B`.

**Qué demuestra este notebook en pantalla (cargando desde Hugging Face, sin Drive):**
1. Verificación automática `verificar_aislamiento_multi_lora()` sobre un único modelo NF4 (los 3 LoRA activos según rol; `Verificador` siempre en pesos Base).
2. Ejecución **en vivo** comparando lado a lado **Control Base (`LoRA OFF`)** vs **D4 `3-LoRA` Oficial** sobre `dificil_25:H07` y **2 consultas naturales de estudiante** (`demo_alumno:A01` certamen médico, `demo_alumno:A02` segunda reprobación en 2.º año).
3. Auditoría transparente de fallos reales de las `75` preguntas (`H03` Recuperador, `H17` Verificador Base, `H06` Redactor) desde `casos_fallo_y_regresion_3lora.json`.

## Paso 1 — Preparación del entorno Colab y clonado de `main` (pre-ejecutar)

In [ ]:
import subprocess
subprocess.run(["nvidia-smi", "-L"], check=False)
%pip -q install -U transformers==5.23.0 peft==0.21.3 bitsandbytes==0.50.2 accelerate==1.15.0 sentence-transformers==6.3.0 huggingface-hub==1.33.0

from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
RAMA = "main"
REPO_DIR = Path("/content/Proyecto_GenIA")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", RAMA], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", RAMA], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", RAMA], check=True)

MONTAR_DRIVE = False  # False = reproduce directamente desde Hugging Face; cambia a True solo para pruebas locales pre-publicación
if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

D4_DIR = REPO_DIR / "Deliverable 4"
if str(D4_DIR / "scripts") not in sys.path:
    sys.path.insert(0, str(D4_DIR / "scripts"))
if str(D4_DIR) not in sys.path:
    sys.path.insert(0, str(D4_DIR))
if str(REPO_DIR / "Deliverable 3" / "Deliverable3" / "agentes") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "Deliverable 3" / "Deliverable3" / "agentes"))

print("Repo listo en:", D4_DIR)
print("Commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())

## Paso 2 — Cargar un único `Qwen/Qwen3-4B` NF4 + los 3 adaptadores QLoRA y comprobar aislamiento

In [ ]:
import json
import subprocess
from IPython.display import HTML, display
import evaluar_d4 as ed4
import evaluar_multi_lora as eml
import publicar_adaptadores_hf as pah
from entrenar_redactor import MODELO_BASE_DEFAULT
from video.mostrar_demo_d4 import (
    cargar_fallos_y_regresiones,
    render_comparativa_html,
    render_estado_carga_html,
    render_fallos_html,
    render_pregunta_banner_html,
    render_resumen_a100_html,
)

ed4.verificar_prompts_d4()
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
manifest_check = pah.verificar_manifest_en_hf()
fuentes = eml.resolver_fuentes_adaptadores_default(solo_hf=not MONTAR_DRIVE)

corpus = ed4.construir_corpus_d3(use_e5=True)
model, tokenizer, dtype_str, base_commit = eml.cargar_modelo_y_multi_lora(
    MODELO_BASE_DEFAULT,
    fuentes,
    roles_a_cargar=eml.ROLES_OFICIALES_D4,
)
aislamiento = eml.verificar_aislamiento_multi_lora(model, tokenizer, eml.ROLES_OFICIALES_D4)
versiones_live = ed4._resolver_versiones_eval()
display(HTML(render_estado_carga_html(MODELO_BASE_DEFAULT, REPO_COMMIT, dtype_str, fuentes, manifest_check, aislamiento)))

## Paso 3 — Comparativa EN VIVO: Control Base (`LoRA OFF`) vs `D4 3-LoRA` (`H07` + 2 consultas naturales de alumno)


In [ ]:
QID_BENCHMARK = "dificil_25:H07"  # Caso benchmark A100 citado en main.pdf
QIDS_ALUMNOS = {"demo_alumno:A01", "demo_alumno:A02"}  # Consultas naturales de alumno (unidades excluidas de entrenamiento)
GREEDY_DEMO = False  # En false usa T=0.7 y semilla=2026 como eval oficial; cambialo a True si buscas determinismo estricto al grabar

PREGUNTAS_DEMO = ed4.cargar_preguntas_evaluacion(filtro_qids={QID_BENCHMARK})
if QIDS_ALUMNOS:
    PREGUNTAS_DEMO += ed4.cargar_preguntas_desde_csv(
        D4_DIR / "datos" / "preguntas_demo_alumnos.csv",
        nombre_set="demo_alumno",
        filtro_qids=QIDS_ALUMNOS,
    )

for pregunta_item in PREGUNTAS_DEMO:
    display(HTML(render_pregunta_banner_html(pregunta_item)))

gen_base = eml.MultiLoraGenerator(
    model,
    tokenizer,
    adaptadores_activos={r: None for r in eml.ROLES_ORDEN},
    greedy=GREEDY_DEMO,
    verbose=True,
)
gen_3lora = eml.MultiLoraGenerator(
    model,
    tokenizer,
    adaptadores_activos={
        "enrutador": "enrutador",
        "recuperador": "recuperador",
        "redactor": "redactor",
        "verificador": None,
    },
    greedy=GREEDY_DEMO,
    verbose=True,
)

OUT_LIVE = Path("/content/demo_live_d4_3lora.json")
OUT_META = Path("/content/demo_live_d4_3lora_meta.json")
pares = eml.ejecutar_comparativa_pareada(
    PREGUNTAS_DEMO,
    corpus,
    gen_base,
    gen_3lora,
    "base",
    "finetuned_3lora",
    OUT_LIVE,
)
OUT_META.write_text(
    json.dumps(
        {
            "repo_commit": REPO_COMMIT,
            "modelo_base": MODELO_BASE_DEFAULT,
            "revision_modelo_base": base_commit,
            "compute_dtype": dtype_str,
            "greedy": GREEDY_DEMO,
            "fuentes_hf": {k: {"repo": v[0], "revision": v[1]} for k, v in fuentes.items()},
            "aislamiento": aislamiento,
            "versiones": versiones_live,
        },
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)
for item in pares:
    display(HTML(render_comparativa_html(item)))

## Paso 4 — Transparencia de errores reales en las `75` (`H03`, `H17`, `H06`) y resumen A100

In [ ]:
fallos = cargar_fallos_y_regresiones(D4_DIR / "evaluacion" / "casos_fallo_y_regresion_3lora.json")
resumen = json.loads((D4_DIR / "evaluacion" / "resumen_comparativo_d4.json").read_text(encoding="utf-8"))
display(HTML(render_fallos_html(fallos, qids_filtro=("dificil_25:H03", "dificil_25:H17", "dificil_25:H06")) + render_resumen_a100_html(resumen)))